In [59]:
import pandas as pd
import numpy as np

from xgboost import XGBClassifier

from sklearn.metrics import brier_score_loss

In [60]:
men = pd.read_csv("../data/processed/men_processed.csv")
women = pd.read_csv("../data/processed/women_processed.csv")

print("Men shape:", men.shape)
print("Women shape:", women.shape)

display(men.head())
display(women.head())

Men shape: (1449, 8)
Women shape: (961, 8)


,Season,Team1,Team2,SeedDiff,AvgScoringMarginDiff,WinRateDiff,EFGDifferentialDiff,Target
0,2003,1411,1421,0,9.208046,0.151724,0.045943,0
1,2003,1112,1436,-15,10.309113,0.237685,0.044359,1
2,2003,1113,1272,3,-1.896552,-0.172414,-0.024137,1
3,2003,1141,1166,5,-8.805643,-0.085684,-0.020792,1
4,2003,1143,1301,-1,0.324138,0.124138,-0.007780,1


,Season,Team1,Team2,SeedDiff,AvgScoringMarginDiff,WinRateDiff,EFGDifferentialDiff,Target
0,2010,3124,3201,-9,2.371212,-0.099432,0.053358,1
1,2010,3173,3395,-1,-0.730769,0.074359,-0.005113,1
2,2010,3181,3214,-13,8.831250,0.210417,0.054759,1
3,2010,3199,3256,-11,4.431183,0.091398,0.039516,1
4,2010,3207,3265,-7,-0.606061,0.012121,-0.087561,1


In [61]:
print("MEN")
print("Seasons:", men["Season"].min(), "-", men["Season"].max())
print("Missing values:", men.isna().sum().sum())
print("Target values:", sorted(men["Target"].unique()))

print("\nWOMEN")
print("Seasons:", women["Season"].min(), "-", women["Season"].max())
print("Missing values:", women.isna().sum().sum())
print("Target values:", sorted(women["Target"].unique()))

print("Men games per season:")
display(men.groupby("Season").size())

print("\nWomen games per season:")
display(women.groupby("Season").size())

MEN
Seasons: 2003 - 2025
Missing values: 0
Target values: [np.int64(0), np.int64(1)]

WOMEN
Seasons: 2010 - 2025
Missing values: 0
Target values: [np.int64(0), np.int64(1)]
Men games per season:


Season
2003    64
2004    64
2005    64
2006    64
2007    64
2008    64
2009    64
2010    64
2011    67
2012    67
2013    67
2014    67
2015    67
2016    67
2017    67
2018    67
2019    67
2021    66
2022    67
2023    67
2024    67
2025    67
dtype: int64


Women games per season:


Season
2010    63
2011    63
2012    63
2013    63
2014    63
2015    63
2016    63
2017    63
2018    63
2019    63
2021    63
2022    67
2023    67
2024    67
2025    67
dtype: int64

In [62]:
FEATURES = [
    "SeedDiff",
    "AvgScoringMarginDiff",
    "WinRateDiff",
    "EFGDifferentialDiff"
]

TARGET = "Target"

In [63]:
men_train = men[men["Season"] <= 2021].copy()
men_test = men[
    men["Season"].between(2022, 2025)
].copy()

women_train = women[women["Season"] <= 2021].copy()
women_test = women[
    women["Season"].between(2022, 2025)
].copy()

print("Men training rows:", len(men_train))
print("Men test rows:", len(men_test))

print("Women training rows:", len(women_train))
print("Women test rows:", len(women_test))



Men training rows: 1181
Men test rows: 268
Women training rows: 693
Women test rows: 268


In [64]:
X_train_m = men_train[FEATURES]
y_train_m = men_train[TARGET]

X_test_m = men_test[FEATURES]
y_test_m = men_test[TARGET]

X_train_w = women_train[FEATURES]
y_train_w = women_train[TARGET]

X_test_w = women_test[FEATURES]
y_test_w = women_test[TARGET]

In [65]:
print("Men training seasons:")
print(sorted(men_train["Season"].unique()))

print("\nMen test seasons:")
print(sorted(men_test["Season"].unique()))

print("\nWomen training seasons:")
print(sorted(women_train["Season"].unique()))

print("\nWomen test seasons:")
print(sorted(women_test["Season"].unique()))

assert set(men_train["Season"]).isdisjoint(
    set(men_test["Season"])
)

assert set(women_train["Season"]).isdisjoint(
    set(women_test["Season"])
)

print("No season overlap between training and test sets.")

Men training seasons:
[np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2021)]

Men test seasons:
[np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

Women training seasons:
[np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2021)]

Women test seasons:
[np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
No season overlap between training and test sets.


In [66]:
men_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42
)

women_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42
)

In [67]:
men_model.fit(
    X_train_m,
    y_train_m
)

women_model.fit(
    X_train_w,
    y_train_w
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)

In [68]:
men_test_prob = men_model.predict_proba(
    X_test_m
)[:, 1]

women_test_prob = women_model.predict_proba(
    X_test_w
)[:, 1]

In [69]:
men_test_brier = brier_score_loss(
    y_test_m,
    men_test_prob
)

women_test_brier = brier_score_loss(
    y_test_w,
    women_test_prob
)

print("Men test Brier:", men_test_brier)
print("Women test Brier:", women_test_brier)

Men test Brier: 0.21006035635233342
Women test Brier: 0.15079699479291142


In [70]:
men_test_results = men_test[
    ["Season", "Team1", "Team2", "Target"]
].copy()

men_test_results["Pred"] = men_test_prob


women_test_results = women_test[
    ["Season", "Team1", "Team2", "Target"]
].copy()

women_test_results["Pred"] = women_test_prob

def brier_by_season(results):
    scores = []

    for season, group in results.groupby("Season"):
        score = brier_score_loss(
            group["Target"],
            group["Pred"]
        )

        scores.append({
            "Season": season,
            "BrierScore": score,
            "Games": len(group)
        })

    return pd.DataFrame(scores)

men_yearly_brier = brier_by_season(
    men_test_results
)

women_yearly_brier = brier_by_season(
    women_test_results
)

print("MEN")
display(men_yearly_brier)

print("WOMEN")
display(women_yearly_brier)

MEN


,Season,BrierScore,Games
0,2022,0.245734,67
1,2023,0.222656,67
2,2024,0.203574,67
3,2025,0.168278,67


WOMEN


,Season,BrierScore,Games
0,2022,0.178221,67
1,2023,0.172622,67
2,2024,0.120076,67
3,2025,0.132269,67


In [71]:
men_train_prob = men_model.predict_proba(
    X_train_m
)[:, 1]

women_train_prob = women_model.predict_proba(
    X_train_w
)[:, 1]

men_train_brier = brier_score_loss(
    y_train_m,
    men_train_prob
)

women_train_brier = brier_score_loss(
    y_train_w,
    women_train_prob
)

print("MEN")
print("Training Brier:", men_train_brier)
print("Test Brier:", men_test_brier)

print("\nWOMEN")
print("Training Brier:", women_train_brier)
print("Test Brier:", women_test_brier)

MEN
Training Brier: 0.16296801381252676
Test Brier: 0.21006035635233342

WOMEN
Training Brier: 0.11486088342869272
Test Brier: 0.15079699479291142


In [72]:
baseline_results = pd.DataFrame({
    "Dataset": ["Men", "Women"],
    "TrainPeriod": [
        "2003-2021",
        "2010-2021"
    ],
    "ValidationPeriod": [
        "2022-2025",
        "2022-2025"
    ],
    "NumFeatures": [4, 4],
    "TrainingBrier": [
        men_train_brier,
        women_train_brier
    ],
    "TestBrier": [
        men_test_brier,
        women_test_brier
    ]
})

display(baseline_results)

,Dataset,TrainPeriod,ValidationPeriod,NumFeatures,TrainingBrier,TestBrier
0,Men,2003-2021,2022-2025,4,0.162968,0.210060
1,Women,2010-2021,2022-2025,4,0.114861,0.150797


## Expanding-Window Validation

The 2022–2025 seasons are reserved as the final historical test set.

Hyperparameter selection is performed using earlier tournament seasons
through expanding-window validation. For each validation season, the model
is trained only using tournament seasons occurring before that season.

Validation seasons: 2017, 2018, 2019 and 2021.
The 2020 season is absent because the NCAA tournament was cancelled.

In [73]:
VALIDATION_YEARS = [
    2017,
    2018,
    2019,
    2021
]

print("Expanding-window validation years:", VALIDATION_YEARS)

Expanding-window validation years: [2017, 2018, 2019, 2021]


In [74]:
def expanding_window_brier(data, validation_years, features, model_params):
    results = []

    for year in validation_years:

        train_fold = data[
            data["Season"] < year
        ].copy()

        valid_fold = data[
            data["Season"] == year
        ].copy()

        X_train = train_fold[features]
        y_train = train_fold[TARGET]

        X_valid = valid_fold[features]
        y_valid = valid_fold[TARGET]

        model = XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=42,
            **model_params
        )

        model.fit(
            X_train,
            y_train
        )

        probabilities = model.predict_proba(
            X_valid
        )[:, 1]

        brier = brier_score_loss(
            y_valid,
            probabilities
        )

        results.append({
            "ValidationSeason": year,
            "TrainingStart": train_fold["Season"].min(),
            "TrainingEnd": train_fold["Season"].max(),
            "TrainingGames": len(train_fold),
            "ValidationGames": len(valid_fold),
            "BrierScore": brier
        })

    return pd.DataFrame(results)

In [75]:
baseline_params = {
    "n_estimators": 100,
    "max_depth": 3,
    "learning_rate": 0.05,
    "subsample": 0.8
}

In [76]:
men_cv_results = expanding_window_brier(
    men_train,
    VALIDATION_YEARS,
    FEATURES,
    baseline_params
)

women_cv_results = expanding_window_brier(
    women_train,
    VALIDATION_YEARS,
    FEATURES,
    baseline_params
)

print("MEN")
display(men_cv_results)

print("\nWOMEN")
display(women_cv_results)

MEN


,ValidationSeason,TrainingStart,TrainingEnd,TrainingGames,ValidationGames,BrierScore
0,2017,2003,2016,914,67,0.174110
1,2018,2003,2017,981,67,0.218208
2,2019,2003,2018,1048,67,0.169112
3,2021,2003,2019,1115,66,0.223271



WOMEN


,ValidationSeason,TrainingStart,TrainingEnd,TrainingGames,ValidationGames,BrierScore
0,2017,2010,2016,441,63,0.170579
1,2018,2010,2017,504,63,0.181549
2,2019,2010,2018,567,63,0.132917
3,2021,2010,2019,630,63,0.133642


In [77]:
men_cv_mean = men_cv_results[
    "BrierScore"
].mean()

women_cv_mean = women_cv_results[
    "BrierScore"
].mean()

print("Men mean expanding-window Brier:", men_cv_mean)
print("Women mean expanding-window Brier:", women_cv_mean)

Men mean expanding-window Brier: 0.19617536763630072
Women mean expanding-window Brier: 0.15467158984217363


## Hyperparameter Tuning

Hyperparameters are selected using mean Brier score from expanding-window
validation. The 2022–2025 test period is not used during tuning.

### 1. max_depth

In [78]:
def tune_parameter(
    data,
    validation_years,
    features,
    base_params,
    parameter_name,
    parameter_values
):
    results = []

    for value in parameter_values:

        params = base_params.copy()
        params[parameter_name] = value

        cv_results = expanding_window_brier(
            data,
            validation_years,
            features,
            params
        )

        mean_brier = cv_results["BrierScore"].mean()

        results.append({
            parameter_name: value,
            "MeanBrier": mean_brier
        })

    return pd.DataFrame(results)

In [79]:
MAX_DEPTH_VALUES = [1, 2, 3, 4, 5]

men_depth_results = tune_parameter(
    men_train,
    VALIDATION_YEARS,
    FEATURES,
    baseline_params,
    "max_depth",
    MAX_DEPTH_VALUES
)

display(men_depth_results)

,max_depth,MeanBrier
0,1,0.199812
1,2,0.197877
2,3,0.196175
3,4,0.193687
4,5,0.195887


In [80]:
women_depth_results = tune_parameter(
    women_train,
    VALIDATION_YEARS,
    FEATURES,
    baseline_params,
    "max_depth",
    MAX_DEPTH_VALUES
)

display(women_depth_results)

,max_depth,MeanBrier
0,1,0.150094
1,2,0.152807
2,3,0.154672
3,4,0.158592
4,5,0.163237


In [81]:
best_men_depth_row = men_depth_results.loc[
    men_depth_results["MeanBrier"].idxmin()
]

best_women_depth_row = women_depth_results.loc[
    women_depth_results["MeanBrier"].idxmin()
]

print("Best men's max_depth:")
print(best_men_depth_row)

print("\nBest women's max_depth:")
print(best_women_depth_row)

Best men's max_depth:
max_depth    4.000000
MeanBrier    0.193687
Name: 3, dtype: float64

Best women's max_depth:
max_depth    1.000000
MeanBrier    0.150094
Name: 0, dtype: float64


In [82]:
men_current_params = {
    "n_estimators": 100,
    "max_depth": 4,
    "learning_rate": 0.05,
    "subsample": 0.8
}

women_current_params = {
    "n_estimators": 100,
    "max_depth": 1,
    "learning_rate": 0.05,
    "subsample": 0.8
}

### 2. learning_rate and n_estimators

Because learning rate controls the contribution of each boosting tree and
n_estimators controls the number of boosting rounds, these parameters are
evaluated jointly.

The best max_depth identified previously is retained separately for the
men's and women's models.

In [83]:
def tune_learning_rate_estimators(
    data,
    validation_years,
    features,
    base_params,
    learning_rates,
    estimator_values
):
    results = []

    for learning_rate in learning_rates:
        for n_estimators in estimator_values:

            params = base_params.copy()

            params["learning_rate"] = learning_rate
            params["n_estimators"] = n_estimators

            cv_results = expanding_window_brier(
                data,
                validation_years,
                features,
                params
            )

            mean_brier = cv_results["BrierScore"].mean()

            results.append({
                "learning_rate": learning_rate,
                "n_estimators": n_estimators,
                "MeanBrier": mean_brier
            })

    return pd.DataFrame(results)

In [84]:
LEARNING_RATES = [0.01, 0.03, 0.05, 0.1]

N_ESTIMATORS_VALUES = [50, 100, 200, 300, 500]

In [85]:
men_lr_estimator_results = tune_learning_rate_estimators(
    men_train,
    VALIDATION_YEARS,
    FEATURES,
    men_current_params,
    LEARNING_RATES,
    N_ESTIMATORS_VALUES
)

men_lr_estimator_results = men_lr_estimator_results.sort_values(
    "MeanBrier"
).reset_index(drop=True)

display(men_lr_estimator_results)

print("Best men's learning_rate / n_estimators:")
display(men_lr_estimator_results.head(5))

,learning_rate,n_estimators,MeanBrier
0,0.01,500,0.192731
1,0.01,300,0.193153
2,0.10,50,0.193281
3,0.05,100,0.193687
4,0.01,200,0.194239
5,0.03,100,0.194432
6,0.05,50,0.194745
7,0.03,200,0.195128
8,0.03,50,0.196537
9,0.10,100,0.196949


Best men's learning_rate / n_estimators:


,learning_rate,n_estimators,MeanBrier
0,0.01,500,0.192731
1,0.01,300,0.193153
2,0.10,50,0.193281
3,0.05,100,0.193687
4,0.01,200,0.194239


In [86]:
women_lr_estimator_results = tune_learning_rate_estimators(
    women_train,
    VALIDATION_YEARS,
    FEATURES,
    women_current_params,
    LEARNING_RATES,
    N_ESTIMATORS_VALUES
)

women_lr_estimator_results = women_lr_estimator_results.sort_values(
    "MeanBrier"
).reset_index(drop=True)

display(women_lr_estimator_results)

print("Best women's learning_rate / n_estimators:")
display(women_lr_estimator_results.head(5))

,learning_rate,n_estimators,MeanBrier
0,0.10,50,0.149545
1,0.05,100,0.150094
2,0.10,100,0.150122
3,0.03,200,0.150219
4,0.01,500,0.150252
5,0.03,300,0.150385
6,0.05,200,0.150828
7,0.05,300,0.151657
8,0.03,500,0.151890
9,0.01,300,0.152024


Best women's learning_rate / n_estimators:


,learning_rate,n_estimators,MeanBrier
0,0.10,50,0.149545
1,0.05,100,0.150094
2,0.10,100,0.150122
3,0.03,200,0.150219
4,0.01,500,0.150252


In [87]:
MEN_REFINED_LEARNING_RATES = [0.005, 0.01, 0.02]

MEN_REFINED_ESTIMATORS = [300, 500, 700, 1000]

men_refined_lr_results = tune_learning_rate_estimators(
    men_train,
    VALIDATION_YEARS,
    FEATURES,
    men_current_params,
    MEN_REFINED_LEARNING_RATES,
    MEN_REFINED_ESTIMATORS
)

men_refined_lr_results = men_refined_lr_results.sort_values(
    "MeanBrier"
).reset_index(drop=True)

display(men_refined_lr_results.head(10))

,learning_rate,n_estimators,MeanBrier
0,0.010,500,0.192731
1,0.010,300,0.193153
2,0.005,500,0.193482
3,0.020,300,0.193505
4,0.005,700,0.193686
5,0.005,1000,0.193731
6,0.010,700,0.194197
7,0.005,300,0.195584
8,0.010,1000,0.195584
9,0.020,500,0.196180


In [88]:
WOMEN_REFINED_LEARNING_RATES = [0.05, 0.1, 0.15, 0.2]

WOMEN_REFINED_ESTIMATORS = [25, 50, 75, 100]

women_refined_lr_results = tune_learning_rate_estimators(
    women_train,
    VALIDATION_YEARS,
    FEATURES,
    women_current_params,
    WOMEN_REFINED_LEARNING_RATES,
    WOMEN_REFINED_ESTIMATORS
)

women_refined_lr_results = women_refined_lr_results.sort_values(
    "MeanBrier"
).reset_index(drop=True)

display(women_refined_lr_results.head(10))

,learning_rate,n_estimators,MeanBrier
0,0.20,25,0.149225
1,0.20,50,0.149415
2,0.10,50,0.149545
3,0.05,100,0.150094
4,0.10,75,0.150098
5,0.10,100,0.150122
6,0.05,75,0.150394
7,0.15,25,0.150765
8,0.15,50,0.150933
9,0.20,75,0.152035


In [89]:
WOMEN_FINAL_LEARNING_RATES = [
    0.15,
    0.20,
    0.25,
    0.30
]

WOMEN_FINAL_ESTIMATORS = [
    10,
    15,
    25,
    35,
    50
]

women_final_lr_results = tune_learning_rate_estimators(
    women_train,
    VALIDATION_YEARS,
    FEATURES,
    women_current_params,
    WOMEN_FINAL_LEARNING_RATES,
    WOMEN_FINAL_ESTIMATORS
)

women_final_lr_results = women_final_lr_results.sort_values(
    "MeanBrier"
).reset_index(drop=True)

display(women_final_lr_results.head(10))

,learning_rate,n_estimators,MeanBrier
0,0.25,25,0.148999
1,0.25,35,0.149027
2,0.20,25,0.149225
3,0.20,50,0.149415
4,0.20,35,0.149880
5,0.30,25,0.149917
6,0.15,35,0.150536
7,0.15,25,0.150765
8,0.25,50,0.150775
9,0.15,50,0.150933


In [90]:
# Current best parameters after tuning:
# 1. max_depth
# 2. learning_rate + n_estimators

men_current_params = {
    "n_estimators": 500,
    "max_depth": 4,
    "learning_rate": 0.01,
    "subsample": 0.8
}

women_current_params = {
    "n_estimators": 25,
    "max_depth": 1,
    "learning_rate": 0.25,
    "subsample": 0.8
}

print("Current men's parameters:")
print(men_current_params)

print("\nCurrent women's parameters:")
print(women_current_params)

Current men's parameters:
{'n_estimators': 500, 'max_depth': 4, 'learning_rate': 0.01, 'subsample': 0.8}

Current women's parameters:
{'n_estimators': 25, 'max_depth': 1, 'learning_rate': 0.25, 'subsample': 0.8}


### 3. subsample

The subsample parameter controls the proportion of training observations
used to construct each boosting tree.

Values below 1.0 introduce randomness that may reduce overfitting, while
values that are too small may prevent each tree from learning enough from
the relatively small tournament dataset.

The previously selected max_depth, learning_rate and n_estimators values
are retained separately for the men's and women's models.

In [91]:
SUBSAMPLE_VALUES = [
    0.6,
    0.7,
    0.8,
    0.9,
    1.0
]

men_subsample_results = tune_parameter(
    men_train,
    VALIDATION_YEARS,
    FEATURES,
    men_current_params,
    "subsample",
    SUBSAMPLE_VALUES
)

women_subsample_results = tune_parameter(
    women_train,
    VALIDATION_YEARS,
    FEATURES,
    women_current_params,
    "subsample",
    SUBSAMPLE_VALUES
)

men_subsample_results = men_subsample_results.sort_values(
    "MeanBrier"
).reset_index(drop=True)

women_subsample_results = women_subsample_results.sort_values(
    "MeanBrier"
).reset_index(drop=True)

print("MEN")
display(men_subsample_results)

print("\nWOMEN")
display(women_subsample_results)

MEN


,subsample,MeanBrier
0,0.8,0.192731
1,1.0,0.193314
2,0.6,0.193615
3,0.7,0.193818
4,0.9,0.193892



WOMEN


,subsample,MeanBrier
0,0.6,0.147115
1,0.7,0.148531
2,0.8,0.148999
3,0.9,0.150800
4,1.0,0.152632


In [92]:
WOMEN_REFINED_SUBSAMPLE_VALUES = [
    0.4,
    0.5,
    0.6,
    0.7
]

women_refined_subsample_results = tune_parameter(
    women_train,
    VALIDATION_YEARS,
    FEATURES,
    women_current_params,
    "subsample",
    WOMEN_REFINED_SUBSAMPLE_VALUES
)

women_refined_subsample_results = women_refined_subsample_results.sort_values(
    "MeanBrier"
).reset_index(drop=True)

display(women_refined_subsample_results)

,subsample,MeanBrier
0,0.4,0.146468
1,0.6,0.147115
2,0.7,0.148531
3,0.5,0.150220


In [93]:
WOMEN_FINAL_SUBSAMPLE_VALUES = [
    0.2,
    0.3,
    0.4,
    0.45,
    0.5
]

women_final_subsample_results = tune_parameter(
    women_train,
    VALIDATION_YEARS,
    FEATURES,
    women_current_params,
    "subsample",
    WOMEN_FINAL_SUBSAMPLE_VALUES
)

women_final_subsample_results = women_final_subsample_results.sort_values(
    "MeanBrier"
).reset_index(drop=True)

display(women_final_subsample_results)

,subsample,MeanBrier
0,0.40,0.146468
1,0.45,0.148256
2,0.20,0.149637
3,0.50,0.150220
4,0.30,0.152827


In [94]:
# Current best parameters after tuning subsample

men_current_params = {
    "n_estimators": 500,
    "max_depth": 4,
    "learning_rate": 0.01,
    "subsample": 0.8
}

women_current_params = {
    "n_estimators": 25,
    "max_depth": 1,
    "learning_rate": 0.25,
    "subsample": 0.4
}

print("Current men's parameters:")
print(men_current_params)

print("\nCurrent women's parameters:")
print(women_current_params)

Current men's parameters:
{'n_estimators': 500, 'max_depth': 4, 'learning_rate': 0.01, 'subsample': 0.8}

Current women's parameters:
{'n_estimators': 25, 'max_depth': 1, 'learning_rate': 0.25, 'subsample': 0.4}


### 4. L2 Regularization (`reg_lambda`)

The `reg_lambda` parameter applies L2 regularization to XGBoost leaf
weights. Larger values shrink leaf contributions more strongly, which may
reduce overfitting but can also cause underfitting if regularization is
excessive.

The previously selected max_depth, learning_rate, n_estimators and
subsample values are retained while reg_lambda is varied.

In [95]:
REG_LAMBDA_VALUES = [
    0,
    0.5,
    1,
    2,
    5,
    10
]

men_lambda_results = tune_parameter(
    men_train,
    VALIDATION_YEARS,
    FEATURES,
    men_current_params,
    "reg_lambda",
    REG_LAMBDA_VALUES
)

women_lambda_results = tune_parameter(
    women_train,
    VALIDATION_YEARS,
    FEATURES,
    women_current_params,
    "reg_lambda",
    REG_LAMBDA_VALUES
)

men_lambda_results = men_lambda_results.sort_values(
    "MeanBrier"
).reset_index(drop=True)

women_lambda_results = women_lambda_results.sort_values(
    "MeanBrier"
).reset_index(drop=True)

print("MEN")
display(men_lambda_results)

print("\nWOMEN")
display(women_lambda_results)

MEN


,reg_lambda,MeanBrier
0,1.0,0.192731
1,0.5,0.193222
2,0.0,0.193334
3,2.0,0.194413
4,5.0,0.194604
5,10.0,0.195357



WOMEN


,reg_lambda,MeanBrier
0,1.0,0.146468
1,0.5,0.147630
2,0.0,0.148083
3,2.0,0.148996
4,5.0,0.149412
5,10.0,0.150747


### 5. L1 Regularization (`reg_alpha`)

The `reg_alpha` parameter applies L1 regularization to XGBoost leaf
weights. Increasing reg_alpha can suppress weak leaf contributions and
reduce model complexity.

The default value is 0, corresponding to no L1 regularization. Candidate
values are evaluated while retaining the previously selected
hyperparameters and reg_lambda=1.

In [96]:
REG_ALPHA_VALUES = [
    0,
    0.01,
    0.05,
    0.1,
    0.5,
    1,
    2
]

men_alpha_results = tune_parameter(
    men_train,
    VALIDATION_YEARS,
    FEATURES,
    men_current_params,
    "reg_alpha",
    REG_ALPHA_VALUES
)

women_alpha_results = tune_parameter(
    women_train,
    VALIDATION_YEARS,
    FEATURES,
    women_current_params,
    "reg_alpha",
    REG_ALPHA_VALUES
)

men_alpha_results = men_alpha_results.sort_values(
    "MeanBrier"
).reset_index(drop=True)

women_alpha_results = women_alpha_results.sort_values(
    "MeanBrier"
).reset_index(drop=True)

print("MEN")
display(men_alpha_results)

print("\nWOMEN")
display(women_alpha_results)

MEN


,reg_alpha,MeanBrier
0,0.00,0.192731
1,0.05,0.192959
2,0.01,0.193038
3,0.10,0.193203
4,0.50,0.194447
5,1.00,0.194654
6,2.00,0.195257



WOMEN


,reg_alpha,MeanBrier
0,0.00,0.146468
1,0.01,0.146473
2,0.05,0.146563
3,0.10,0.146588
4,0.50,0.148239
5,1.00,0.150538
6,2.00,0.150677


### 6. Combined Hyperparameter Search

The preceding experiments varied individual hyperparameters or closely
related parameter pairs while holding the remaining settings fixed.

Because XGBoost hyperparameters can interact, a focused combined search is
now performed around the promising regions identified during sequential
tuning.

The 2022–2025 holdout period remains excluded from all parameter selection.

In [97]:
from itertools import product


def combined_parameter_search(
    data,
    validation_years,
    features,
    depths,
    learning_rates,
    estimator_values,
    subsample_values
):
    results = []

    for depth, learning_rate, n_estimators, subsample in product(
        depths,
        learning_rates,
        estimator_values,
        subsample_values
    ):

        params = {
            "max_depth": depth,
            "learning_rate": learning_rate,
            "n_estimators": n_estimators,
            "subsample": subsample,
            "reg_lambda": 1,
            "reg_alpha": 0
        }

        cv_results = expanding_window_brier(
            data,
            validation_years,
            features,
            params
        )

        mean_brier = cv_results["BrierScore"].mean()

        results.append({
            "max_depth": depth,
            "learning_rate": learning_rate,
            "n_estimators": n_estimators,
            "subsample": subsample,
            "MeanBrier": mean_brier
        })

    return (
        pd.DataFrame(results)
        .sort_values("MeanBrier")
        .reset_index(drop=True)
    )

In [98]:
MEN_COMBINED_DEPTHS = [3, 4, 5]
MEN_COMBINED_LEARNING_RATES = [0.005, 0.01, 0.02]
MEN_COMBINED_ESTIMATORS = [300, 500, 700]
MEN_COMBINED_SUBSAMPLES = [0.7, 0.8, 0.9]


men_combined_results = combined_parameter_search(
    men_train,
    VALIDATION_YEARS,
    FEATURES,
    MEN_COMBINED_DEPTHS,
    MEN_COMBINED_LEARNING_RATES,
    MEN_COMBINED_ESTIMATORS,
    MEN_COMBINED_SUBSAMPLES
)

print("Top 10 men's combined configurations:")
display(men_combined_results.head(10))

Top 10 men's combined configurations:


,max_depth,learning_rate,n_estimators,subsample,MeanBrier
0,5,0.010,300,0.7,0.191705
1,5,0.005,500,0.7,0.192247
2,5,0.010,300,0.8,0.192290
3,5,0.005,500,0.8,0.192417
4,4,0.020,300,0.7,0.192440
5,5,0.010,500,0.7,0.192579
6,5,0.005,700,0.8,0.192666
7,4,0.010,500,0.8,0.192731
8,5,0.005,700,0.7,0.192927
9,4,0.020,500,0.7,0.192950


In [99]:
WOMEN_COMBINED_DEPTHS = [1, 2]
WOMEN_COMBINED_LEARNING_RATES = [0.20, 0.25, 0.30]
WOMEN_COMBINED_ESTIMATORS = [15, 25, 35]
WOMEN_COMBINED_SUBSAMPLES = [0.3, 0.4, 0.5, 0.6]


women_combined_results = combined_parameter_search(
    women_train,
    VALIDATION_YEARS,
    FEATURES,
    WOMEN_COMBINED_DEPTHS,
    WOMEN_COMBINED_LEARNING_RATES,
    WOMEN_COMBINED_ESTIMATORS,
    WOMEN_COMBINED_SUBSAMPLES
)

print("Top 10 women's combined configurations:")
display(women_combined_results.head(10))

Top 10 women's combined configurations:


,max_depth,learning_rate,n_estimators,subsample,MeanBrier
0,1,0.25,25,0.4,0.146468
1,1,0.30,15,0.5,0.146503
2,1,0.25,25,0.6,0.147115
3,1,0.30,25,0.5,0.148033
4,1,0.20,25,0.6,0.148108
5,1,0.20,35,0.6,0.148314
6,1,0.25,15,0.6,0.148597
7,1,0.25,35,0.6,0.148894
8,1,0.25,15,0.4,0.149036
9,1,0.20,25,0.5,0.149109


### 7. Random-Seed Robustness

Because subsampling introduces randomness into XGBoost training, the leading
hyperparameter configurations are evaluated across multiple random seeds.

This checks whether the selected configurations consistently generalize
well or whether their performance depends strongly on a particular random
sampling sequence.

In [100]:
def seed_robustness_test(
    data,
    validation_years,
    features,
    model_params,
    seeds
):
    results = []

    for seed in seeds:

        fold_scores = []

        for year in validation_years:

            train_fold = data[
                data["Season"] < year
            ].copy()

            valid_fold = data[
                data["Season"] == year
            ].copy()

            X_train = train_fold[features]
            y_train = train_fold[TARGET]

            X_valid = valid_fold[features]
            y_valid = valid_fold[TARGET]

            model = XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=seed,
                **model_params
            )

            model.fit(
                X_train,
                y_train
            )

            probabilities = model.predict_proba(
                X_valid
            )[:, 1]

            brier = brier_score_loss(
                y_valid,
                probabilities
            )

            fold_scores.append(brier)

        results.append({
            "Seed": seed,
            "MeanBrier": np.mean(fold_scores)
        })

    return pd.DataFrame(results)

In [101]:
men_combined_best_params = {
    "max_depth": 5,
    "learning_rate": 0.01,
    "n_estimators": 300,
    "subsample": 0.7,
    "reg_lambda": 1,
    "reg_alpha": 0
}

ROBUSTNESS_SEEDS = [
    0,
    1,
    21,
    42,
    100
]

men_seed_results = seed_robustness_test(
    men_train,
    VALIDATION_YEARS,
    FEATURES,
    men_combined_best_params,
    ROBUSTNESS_SEEDS
)

display(men_seed_results)

print(
    "Mean across seeds:",
    men_seed_results["MeanBrier"].mean()
)

print(
    "Standard deviation:",
    men_seed_results["MeanBrier"].std()
)

,Seed,MeanBrier
0,0,0.193142
1,1,0.191944
2,21,0.193161
3,42,0.191705
4,100,0.193201


Mean across seeds: 0.19263039462664974
Standard deviation: 0.0007411164671030891


In [102]:
women_candidate_1 = {
    "max_depth": 1,
    "learning_rate": 0.25,
    "n_estimators": 25,
    "subsample": 0.4,
    "reg_lambda": 1,
    "reg_alpha": 0
}

women_candidate_2 = {
    "max_depth": 1,
    "learning_rate": 0.30,
    "n_estimators": 15,
    "subsample": 0.5,
    "reg_lambda": 1,
    "reg_alpha": 0
}


women_seed_results_1 = seed_robustness_test(
    women_train,
    VALIDATION_YEARS,
    FEATURES,
    women_candidate_1,
    ROBUSTNESS_SEEDS
)

women_seed_results_2 = seed_robustness_test(
    women_train,
    VALIDATION_YEARS,
    FEATURES,
    women_candidate_2,
    ROBUSTNESS_SEEDS
)


print("WOMEN — Candidate 1")
display(women_seed_results_1)

print(
    "Mean across seeds:",
    women_seed_results_1["MeanBrier"].mean()
)

print(
    "Standard deviation:",
    women_seed_results_1["MeanBrier"].std()
)


print("\nWOMEN — Candidate 2")
display(women_seed_results_2)

print(
    "Mean across seeds:",
    women_seed_results_2["MeanBrier"].mean()
)

print(
    "Standard deviation:",
    women_seed_results_2["MeanBrier"].std()
)

WOMEN — Candidate 1


,Seed,MeanBrier
0,0,0.151204
1,1,0.152330
2,21,0.152766
3,42,0.146468
4,100,0.149905


Mean across seeds: 0.15053463667817074
Standard deviation: 0.002529160129418786

WOMEN — Candidate 2


,Seed,MeanBrier
0,0,0.152514
1,1,0.147442
2,21,0.147322
3,42,0.146503
4,100,0.152862


Mean across seeds: 0.14932855375265172
Standard deviation: 0.0030903102783490233


### Final Hyperparameter Selection Before Holdout Evaluation

Following sequential tuning, combined hyperparameter search, and
random-seed robustness analysis, one configuration was selected separately
for the men's and women's models.

For the men's model, the combined-search configuration remained relatively
stable across random seeds.

For the women's model, two nearly tied configurations under random seed 42
were compared across multiple seeds. The second candidate achieved a lower
mean Brier score across seeds and was therefore selected despite slightly
higher variability.

The 2022–2025 holdout period was not used during this selection process.

In [103]:
final_men_params = {
    "max_depth": 5,
    "learning_rate": 0.01,
    "n_estimators": 300,
    "subsample": 0.7,
    "reg_lambda": 1,
    "reg_alpha": 0
}

final_women_params = {
    "max_depth": 1,
    "learning_rate": 0.30,
    "n_estimators": 15,
    "subsample": 0.5,
    "reg_lambda": 1,
    "reg_alpha": 0
}

print("Final men's parameters:")
print(final_men_params)

print("\nFinal women's parameters:")
print(final_women_params)

Final men's parameters:
{'max_depth': 5, 'learning_rate': 0.01, 'n_estimators': 300, 'subsample': 0.7, 'reg_lambda': 1, 'reg_alpha': 0}

Final women's parameters:
{'max_depth': 1, 'learning_rate': 0.3, 'n_estimators': 15, 'subsample': 0.5, 'reg_lambda': 1, 'reg_alpha': 0}


## Final Holdout Evaluation

After hyperparameter selection using expanding-window validation and
random-seed robustness analysis, the selected configurations are evaluated
on the untouched 2022–2025 holdout seasons.

No hyperparameters are modified based on the holdout results.

The models are trained using all available historical tournament examples
through 2021 and evaluated on tournament games from 2022–2025.

In [104]:
final_men_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    **final_men_params
)

final_women_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    **final_women_params
)

# Train using all pre-2022 tournament examples
final_men_model.fit(
    X_train_m,
    y_train_m
)

final_women_model.fit(
    X_train_w,
    y_train_w
)

print("Final models trained successfully.")

Final models trained successfully.


In [105]:
final_men_test_prob = final_men_model.predict_proba(
    X_test_m
)[:, 1]

final_women_test_prob = final_women_model.predict_proba(
    X_test_w
)[:, 1]

In [106]:
final_men_test_brier = brier_score_loss(
    y_test_m,
    final_men_test_prob
)

final_women_test_brier = brier_score_loss(
    y_test_w,
    final_women_test_prob
)

print("FINAL HOLDOUT RESULTS")
print("---------------------")

print(
    "Men baseline Brier:",
    men_test_brier
)

print(
    "Men tuned Brier:",
    final_men_test_brier
)

print()

print(
    "Women baseline Brier:",
    women_test_brier
)

print(
    "Women tuned Brier:",
    final_women_test_brier
)

FINAL HOLDOUT RESULTS
---------------------
Men baseline Brier: 0.21006035635233342
Men tuned Brier: 0.21288967571793474

Women baseline Brier: 0.15079699479291142
Women tuned Brier: 0.14533677633145972


In [107]:
men_improvement = (
    men_test_brier
    - final_men_test_brier
)

women_improvement = (
    women_test_brier
    - final_women_test_brier
)

men_improvement_pct = (
    men_improvement
    / men_test_brier
) * 100

women_improvement_pct = (
    women_improvement
    / women_test_brier
) * 100


print("MEN")
print("Absolute Brier improvement:", men_improvement)
print("Relative improvement (%):", men_improvement_pct)

print("\nWOMEN")
print("Absolute Brier improvement:", women_improvement)
print("Relative improvement (%):", women_improvement_pct)

MEN
Absolute Brier improvement: -0.002829319365601324
Relative improvement (%): -1.3469078196057698

WOMEN
Absolute Brier improvement: 0.005460218461451699
Relative improvement (%): 3.6209066824907112


In [108]:
final_men_test_results = men_test[
    ["Season", "Team1", "Team2", "Target"]
].copy()

final_men_test_results["Pred"] = final_men_test_prob


final_women_test_results = women_test[
    ["Season", "Team1", "Team2", "Target"]
].copy()

final_women_test_results["Pred"] = final_women_test_prob


final_men_yearly = brier_by_season(
    final_men_test_results
)

final_women_yearly = brier_by_season(
    final_women_test_results
)


print("FINAL TUNED MEN — BY SEASON")
display(final_men_yearly)

print("\nFINAL TUNED WOMEN — BY SEASON")
display(final_women_yearly)

FINAL TUNED MEN — BY SEASON


,Season,BrierScore,Games
0,2022,0.242093,67
1,2023,0.227380,67
2,2024,0.208366,67
3,2025,0.173720,67



FINAL TUNED WOMEN — BY SEASON


,Season,BrierScore,Games
0,2022,0.173520,67
1,2023,0.169809,67
2,2024,0.114970,67
3,2025,0.123048,67


In [109]:
holdout_comparison = pd.DataFrame({
    "Dataset": [
        "Men",
        "Women"
    ],

    "BaselineBrier": [
        men_test_brier,
        women_test_brier
    ],

    "TunedBrier": [
        final_men_test_brier,
        final_women_test_brier
    ],

    "AbsoluteImprovement": [
        men_improvement,
        women_improvement
    ],

    "RelativeImprovementPercent": [
        men_improvement_pct,
        women_improvement_pct
    ]
})

display(holdout_comparison)

,Dataset,BaselineBrier,TunedBrier,AbsoluteImprovement,RelativeImprovementPercent
0,Men,0.210060,0.212890,-0.002829,-1.346908
1,Women,0.150797,0.145337,0.005460,3.620907


### Four-Feature XGBoost Holdout Findings

Hyperparameter tuning produced different generalization outcomes for the
men's and women's datasets.

For the men's model, tuning improved expanding-window cross-validation
performance but did not improve the untouched 2022–2025 holdout Brier
score. The tuned model obtained a holdout Brier score of 0.212890 compared
with 0.210060 for the baseline model.

For the women's model, tuning improved both cross-validation and holdout
performance. The holdout Brier score decreased from 0.150797 to 0.145337,
with improvements observed in each holdout season from 2022 through 2025.

The holdout results were not used for further hyperparameter tuning.

In [110]:
men_elo = pd.read_csv("../data/processed/men_elo.csv")
women_elo = pd.read_csv("../data/processed/women_elo.csv")

print("Men Elo:", men_elo.shape)
print("Women Elo:", women_elo.shape)

display(men_elo.head())
display(women_elo.head())

Men Elo: (13753, 3)
Women Elo: (9851, 3)


,Season,TeamID,FinalElo
0,1985,1228,1748.562289
1,1985,1328,1840.722709
2,1985,1417,1638.757980
3,1985,1225,1265.971331
4,1985,1412,1680.784833


,Season,TeamID,FinalElo
0,1998,3104,1916.856967
1,1998,3202,1521.946455
2,1998,3163,2000.588851
3,1998,3221,1775.678821
4,1998,3222,1347.543298


In [112]:
def add_elo_feature(matchups, elo):

    # Team1 Elo
    result = matchups.merge(
        elo.rename(columns={
            "TeamID": "Team1",
            "FinalElo": "Team1Elo"
        }),
        on=["Season", "Team1"],
        how="left"
    )

    # Team2 Elo
    result = result.merge(
        elo.rename(columns={
            "TeamID": "Team2",
            "FinalElo": "Team2Elo"
        }),
        on=["Season", "Team2"],
        how="left"
    )

    # Positive = Team1 has higher Elo
    result["EloDiff"] = (
        result["Team1Elo"] - result["Team2Elo"]
    )

    return result


men_with_elo = add_elo_feature(men, men_elo)
women_with_elo = add_elo_feature(women, women_elo)

In [113]:
def check_elo_merge(df, name):

    print(f"\n{name}")
    print("-" * 40)

    print("Rows:", len(df))
    print("Missing Team1Elo:", df["Team1Elo"].isna().sum())
    print("Missing Team2Elo:", df["Team2Elo"].isna().sum())
    print("Missing EloDiff:", df["EloDiff"].isna().sum())

    print("\nEloDiff summary:")
    display(df["EloDiff"].describe())


check_elo_merge(men_with_elo, "MEN")
check_elo_merge(women_with_elo, "WOMEN")


MEN
----------------------------------------
Rows: 1449
Missing Team1Elo: 0
Missing Team2Elo: 0
Missing EloDiff: 0

EloDiff summary:


count    1449.000000
mean        1.163288
std       133.095121
min      -466.027853
25%       -75.617904
50%         0.033934
75%        84.128741
max       343.972520
Name: EloDiff, dtype: float64


WOMEN
----------------------------------------
Rows: 961
Missing Team1Elo: 0
Missing Team2Elo: 0
Missing EloDiff: 0

EloDiff summary:


count    961.000000
mean       5.085790
std      218.569300
min     -644.030438
25%     -131.234027
50%        2.303558
75%      144.747893
max      630.624356
Name: EloDiff, dtype: float64

In [114]:
FEATURES = [
    "SeedDiff",
    "AvgScoringMarginDiff",
    "WinRateDiff",
    "EFGDifferentialDiff"
]

FEATURES_WITH_ELO = [
    "SeedDiff",
    "AvgScoringMarginDiff",
    "WinRateDiff",
    "EFGDifferentialDiff",
    "EloDiff"
]

print("Original features:", FEATURES)
print("With Elo:", FEATURES_WITH_ELO)

Original features: ['SeedDiff', 'AvgScoringMarginDiff', 'WinRateDiff', 'EFGDifferentialDiff']
With Elo: ['SeedDiff', 'AvgScoringMarginDiff', 'WinRateDiff', 'EFGDifferentialDiff', 'EloDiff']


In [117]:
men_4feature_cv = expanding_window_brier(
    men,
    validation_years=[2017, 2018, 2019, 2021],
    features=FEATURES,
    model_params=final_men_params
)

men_5feature_cv = expanding_window_brier(
    men_with_elo,
    validation_years=[2017, 2018, 2019, 2021],
    features=FEATURES_WITH_ELO,
    model_params=final_men_params
)

women_4feature_cv = expanding_window_brier(
    women,
    validation_years=[2017, 2018, 2019, 2021],
    features=FEATURES,
    model_params=final_women_params
)

women_5feature_cv = expanding_window_brier(
    women_with_elo,
    validation_years=[2017, 2018, 2019, 2021],
    features=FEATURES_WITH_ELO,
    model_params=final_women_params
)

In [118]:
print("MEN")
print("4 features:      ", men_4feature_cv)
print("4 features + Elo:", men_5feature_cv)

print("\nWOMEN")
print("4 features:      ", women_4feature_cv)
print("4 features + Elo:", women_5feature_cv)

MEN
4 features:          ValidationSeason  TrainingStart  TrainingEnd  TrainingGames  \
0              2017           2003         2016            914   
1              2018           2003         2017            981   
2              2019           2003         2018           1048   
3              2021           2003         2019           1115   

   ValidationGames  BrierScore  
0               67    0.164758  
1               67    0.220298  
2               67    0.162818  
3               66    0.218945  
4 features + Elo:    ValidationSeason  TrainingStart  TrainingEnd  TrainingGames  \
0              2017           2003         2016            914   
1              2018           2003         2017            981   
2              2019           2003         2018           1048   
3              2021           2003         2019           1115   

   ValidationGames  BrierScore  
0               67    0.178890  
1               67    0.227639  
2               67    0.167128  
3

### Elo Feature Ablation

Adding EloDiff to the existing four-feature XGBoost model did not improve
temporal cross-validation performance.

- Men: Brier increased from 0.191705 to 0.198060.
- Women: Brier increased from 0.146503 to 0.149180.

The deterioration occurred across most validation seasons, suggesting that
EloDiff did not provide sufficient additional predictive information beyond
the existing seed, scoring-margin, win-rate, and eFG-differential features.
Elo was therefore not automatically retained in the final feature set.

In [119]:
ELO_ONLY_FEATURES = [
    "EloDiff"
]

In [120]:
men_elo_only_cv = expanding_window_brier(
    men_with_elo,
    validation_years=[2017, 2018, 2019, 2021],
    features=ELO_ONLY_FEATURES,
    model_params=final_men_params
)

women_elo_only_cv = expanding_window_brier(
    women_with_elo,
    validation_years=[2017, 2018, 2019, 2021],
    features=ELO_ONLY_FEATURES,
    model_params=final_women_params
)

print("MEN — ELO ONLY")
print(men_elo_only_cv)

print("\nWOMEN — ELO ONLY")
print(women_elo_only_cv)

MEN — ELO ONLY
   ValidationSeason  TrainingStart  TrainingEnd  TrainingGames  \
0              2017           2003         2016            914   
1              2018           2003         2017            981   
2              2019           2003         2018           1048   
3              2021           2003         2019           1115   

   ValidationGames  BrierScore  
0               67    0.233882  
1               67    0.228826  
2               67    0.202869  
3               66    0.250059  

WOMEN — ELO ONLY
   ValidationSeason  TrainingStart  TrainingEnd  TrainingGames  \
0              2017           2010         2016            441   
1              2018           2010         2017            504   
2              2019           2010         2018            567   
3              2021           2010         2019            630   

   ValidationGames  BrierScore  
0               63    0.206819  
1               63    0.194636  
2               63    0.178995  
3       

In [ ]:
### Elo Feature Ablation

Elo ratings were generated from regular-season games using temporally
validated K-factors of 60 for men and 100 for women.

The contribution of EloDiff was evaluated using the same expanding-window
tournament validation procedure and fixed XGBoost configurations.

| Feature Set | Men Mean Brier | Women Mean Brier |
|---|---:|---:|
| Original 4 Features | 0.191705 | 0.146503 |
| Original 4 + EloDiff | 0.198060 | 0.149180 |
| EloDiff Only | 0.228909 | 0.203368 |

EloDiff alone provided predictive information relative to an uninformed
0.5 probability baseline (Brier = 0.25), but was substantially weaker than
the original four-feature set. Adding EloDiff to the existing features also
increased validation Brier for both men's and women's models.

Therefore, EloDiff was not retained in the current feature set.

In [121]:
men_massey = pd.read_csv(
    "../data/processed/men_massey.csv"
)

print("Shape:", men_massey.shape)

print(
    "Missing MasseyRankDiff:",
    men_massey["MasseyRankDiff"].isna().sum()
)

display(men_massey.head())

Shape: (1449, 11)
Missing MasseyRankDiff: 0


,Season,Team1,Team2,SeedDiff,AvgScoringMarginDiff,WinRateDiff,EFGDifferentialDiff,Target,Team1MasseyRank,Team2MasseyRank,MasseyRankDiff
0,2003,1411,1421,0,9.208046,0.151724,0.045943,0,250.4,251.6,-1.2
1,2003,1112,1436,-15,10.309113,0.237685,0.044359,1,2.6,156.8,-154.2
2,2003,1113,1272,3,-1.896552,-0.172414,-0.024137,1,34.8,20.8,14.0
3,2003,1141,1166,5,-8.805643,-0.085684,-0.020792,1,40.8,23.0,17.8
4,2003,1143,1301,-1,0.324138,0.124138,-0.007780,1,33.8,47.0,-13.2


In [122]:
FEATURES_WITH_MASSEY = [
    "SeedDiff",
    "AvgScoringMarginDiff",
    "WinRateDiff",
    "EFGDifferentialDiff",
    "MasseyRankDiff"
]

In [123]:
men_4feature_massey_comparison = expanding_window_brier(
    men,
    validation_years=[2017, 2018, 2019, 2021],
    features=FEATURES,
    model_params=final_men_params
)

men_5feature_massey_cv = expanding_window_brier(
    men_massey,
    validation_years=[2017, 2018, 2019, 2021],
    features=FEATURES_WITH_MASSEY,
    model_params=final_men_params
)

print("MEN — 4 FEATURES")
print(men_4feature_massey_comparison)

print("\nMEN — 4 FEATURES + MASSEY")
print(men_5feature_massey_cv)

MEN — 4 FEATURES
   ValidationSeason  TrainingStart  TrainingEnd  TrainingGames  \
0              2017           2003         2016            914   
1              2018           2003         2017            981   
2              2019           2003         2018           1048   
3              2021           2003         2019           1115   

   ValidationGames  BrierScore  
0               67    0.164758  
1               67    0.220298  
2               67    0.162818  
3               66    0.218945  

MEN — 4 FEATURES + MASSEY
   ValidationSeason  TrainingStart  TrainingEnd  TrainingGames  \
0              2017           2003         2016            914   
1              2018           2003         2017            981   
2              2019           2003         2018           1048   
3              2021           2003         2019           1115   

   ValidationGames  BrierScore  
0               67    0.169810  
1               67    0.210815  
2               67    0.155767

In [124]:
MASSEY_ONLY_FEATURES = [
    "MasseyRankDiff"
]

men_massey_only_cv = expanding_window_brier(
    men_massey,
    validation_years=[2017, 2018, 2019, 2021],
    features=MASSEY_ONLY_FEATURES,
    model_params=final_men_params
)

print("MEN — MASSEY ONLY")
print(men_massey_only_cv)

MEN — MASSEY ONLY
   ValidationSeason  TrainingStart  TrainingEnd  TrainingGames  \
0              2017           2003         2016            914   
1              2018           2003         2017            981   
2              2019           2003         2018           1048   
3              2021           2003         2019           1115   

   ValidationGames  BrierScore  
0               67    0.185636  
1               67    0.201676  
2               67    0.165651  
3               66    0.236498  


In [125]:
FEATURES_WITH_MASSEY_ELO = [
    "SeedDiff",
    "AvgScoringMarginDiff",
    "WinRateDiff",
    "EFGDifferentialDiff",
    "MasseyRankDiff",
    "EloDiff"
]

In [126]:
men_massey_elo = men_massey.merge(
    men_elo[["Season", "TeamID", "FinalElo"]].rename(
        columns={
            "TeamID": "Team1",
            "FinalElo": "Team1Elo"
        }
    ),
    on=["Season", "Team1"],
    how="left"
)

men_massey_elo = men_massey_elo.merge(
    men_elo[["Season", "TeamID", "FinalElo"]].rename(
        columns={
            "TeamID": "Team2",
            "FinalElo": "Team2Elo"
        }
    ),
    on=["Season", "Team2"],
    how="left"
)

men_massey_elo["EloDiff"] = (
    men_massey_elo["Team1Elo"]
    - men_massey_elo["Team2Elo"]
)

print("Rows:", len(men_massey_elo))
print(
    "Missing EloDiff:",
    men_massey_elo["EloDiff"].isna().sum()
)

Rows: 1449
Missing EloDiff: 0


In [127]:
men_massey_elo_cv = expanding_window_brier(
    men_massey_elo,
    validation_years=[2017, 2018, 2019, 2021],
    features=FEATURES_WITH_MASSEY_ELO,
    model_params=final_men_params
)

print("MEN — 4 FEATURES + MASSEY + ELO")
print(men_massey_elo_cv)

MEN — 4 FEATURES + MASSEY + ELO
   ValidationSeason  TrainingStart  TrainingEnd  TrainingGames  \
0              2017           2003         2016            914   
1              2018           2003         2017            981   
2              2019           2003         2018           1048   
3              2021           2003         2019           1115   

   ValidationGames  BrierScore  
0               67    0.185444  
1               67    0.217701  
2               67    0.162479  
3               66    0.222478  


In [ ]:
### Men's Feature Ablation

Additional Elo and Massey ranking features were evaluated using the same
expanding-window temporal cross-validation procedure and fixed XGBoost
configuration.

| Feature Set | Mean CV Brier |
|---|---:|
| Original 4 Features | 0.191705 |
| EloDiff Only | 0.228909 |
| MasseyRankDiff Only | 0.197365 |
| Original 4 + EloDiff | 0.198060 |
| Original 4 + MasseyRankDiff | 0.189886 |
| Original 4 + MasseyRankDiff + EloDiff | 0.197026 |

EloDiff increased validation Brier both with and without MasseyRankDiff and
was therefore excluded from the candidate feature set.

MasseyRankDiff alone was weaker than the original four-feature model, but
adding it to the original features reduced mean Brier from 0.191705 to
0.189886. This suggests that the consensus Massey ranking provided some
complementary information to the existing features, although the improvement
was modest and varied across validation seasons.